## 환경 설정 및 라이브러리 임포트

In [ ]:
import os
import re
import cv2
import uuid
import time
import json
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from dotenv import load_dotenv
from pathlib import Path

env_path = Path("C:/Users/Administrator/Documents/AH_03_06/.env.development")
load_dotenv(dotenv_path=env_path)

SECRET_KEY = os.getenv("CLOVA_OCR_SECRET_KEY")
INVOKE_URL  = os.getenv("CLOVA_OCR_INVOKE_URL")

print(f"SECRET_KEY: {'설정됨' if SECRET_KEY else '없음'}")
print(f"INVOKE_URL: {'설정됨' if INVOKE_URL else '없음'}")

## 경로 설정

In [ ]:
BASE_DIR   = "../../data/ocr/prescriptions"
INPUT_DIR  = os.path.join(BASE_DIR, "output_png")
AUG_DIR    = os.path.join(BASE_DIR, "output_aug")
RESULT_DIR = os.path.join(BASE_DIR, "output_ocr")
CSV_PATH   = os.path.join(BASE_DIR, "처방전_데이터.csv")
os.makedirs(RESULT_DIR, exist_ok=True)

df_gt = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print(f"원본: {INPUT_DIR}")
print(f"증강: {AUG_DIR}")
print(f"결과: {RESULT_DIR}")
print(f"정답 CSV: {len(df_gt)}개")

## Clova OCR API 함수

In [ ]:
def call_clova_ocr(img_path):
    """Clova OCR API 호출 → 결과 JSON 반환"""
    with open(img_path, "rb") as f:
        img_data = f.read()

    ext = os.path.splitext(img_path)[1].lower().replace(".", "")
    if ext == "jpg":
        ext = "jpeg"

    payload = {
        "images": [{"format": ext, "name": "prescription"}],
        "requestId": str(uuid.uuid4()),
        "version": "V2",
        "timestamp": int(time.time() * 1000),
    }

    response = requests.post(
        INVOKE_URL,
        headers={"X-OCR-SECRET": SECRET_KEY},
        data={"message": json.dumps(payload)},
        files={"file": img_data},
    )

    if response.status_code != 200:
        print(f"에러: {response.status_code} {response.text}")
        return None

    return response.json()


def parse_clova_result(result):
    """Clova OCR 결과 → 텍스트 리스트 변환"""
    if not result:
        return []
    texts = []
    for image in result.get("images", []):
        for field in image.get("fields", []):
            texts.append({
                "text": field["inferText"],
                "confidence": field["inferConfidence"],
                "bbox": field["boundingPoly"]["vertices"],
            })
    return texts

print("API 함수 정의 완료")

## 필드 추출 함수

In [ ]:
def extract_fields(texts):
    """Clova OCR 텍스트 리스트 → 처방전 필드 딕셔너리"""
    text_list = [t["text"] for t in texts]

    fields = {
        "facility_code": None,
        "facility_name": None,
        "patient_name":  None,
        "patient_ssn":   None,
        "disease_code":  None,
        "doctor_name":   None,
        "medications":   [],
        "inject_info":   [],
    }

    skip_words = [
        "환자", "의사", "의료", "처방", "발급", "성명", "기관", "분류",
        "면허", "전화번호", "팩스번호", "주민등록번호", "요양기관기호", "연월일",
        "성", "명", "의원",
    ]

    for i, text in enumerate(text_list):

        # 요양기관기호 (텍스트 내 8자리 숫자 추출 또는 다음 텍스트)
        if "요양기관기호" in text:
            match = re.search(r'\d{8}', text)
            if match:
                fields["facility_code"] = match.group()
            elif i+1 < len(text_list):
                fields["facility_code"] = text_list[i+1]

        # 병원명 (명칭 다음 텍스트)
        if text == "명칭" and i+1 < len(text_list):
            candidate = text_list[i+1]
            if re.search(r"[가-힣]", candidate) and any(kw in candidate for kw in ["의원", "병원", "의학", "클리닉"]):
                fields["facility_name"] = candidate

        # 환자명 (주민등록번호 앞 최대 10칸에서 한글 이름 탐색)
        if "주민등록번호" in text:
            for j in range(i-1, max(i-10, -1), -1):
                candidate = text_list[j]
                if re.match(r"^[가-힣]{2,4}$", candidate) and candidate not in skip_words:
                    fields["patient_name"] = candidate
                    break
            if i+1 < len(text_list):
                fields["patient_ssn"] = text_list[i+1]

        # 질병코드 (알파벳+숫자 패턴)
        if re.match(r"^[A-Z]\d{2}", text):
            fields["disease_code"] = text

        # 의사명 (성명 다음 한글 이름)
        if text == "성명" and i+1 < len(text_list):
            candidate = text_list[i+1]
            if re.match(r"^[가-힣]{2,4}$", candidate) and candidate not in skip_words:
                fields["doctor_name"] = candidate

        # 약 이름 (한글 시작 + mg/캡슐/정으로 끝나는 텍스트)
        if re.search(r"(mg|캡슐|정)\d*$", text, re.IGNORECASE):
            if re.match(r"^[가-힣]", text) and text not in fields["medications"]:
                fields["medications"].append(text)

        # 주사제 이름
        if "주사" in text and "주사제" not in text and "처방명세" not in text:
            fields["inject_info"].append(text)

    return fields

print("필드 추출 함수 정의 완료")

## 평가 함수

In [ ]:
def normalize(text):
    """비교를 위한 텍스트 정규화"""
    if not text or pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[^\w가-힣]", "", text)
    return text.lower()


def evaluate(fields, facility_code):
    """추출된 필드를 CSV 정답과 비교해 정확도 반환"""
    gt_row = df_gt[df_gt["facility_code"].astype(str) == str(facility_code)]
    if len(gt_row) == 0:
        print(f"정답 못 찾음: facility_code={facility_code}")
        return None

    gt = gt_row.iloc[0]

    comparisons = {
        "facility_code": (fields.get("facility_code"),   str(gt["facility_code"])),
        "facility_name": (fields.get("facility_name"),   str(gt["facility_name"])),
        "patient_name":  (fields.get("patient_name"),    str(gt["patient_name"])),
        "patient_ssn":   (fields.get("patient_ssn"),     str(gt["patient_ssn"])),
        "disease_code":  (fields.get("disease_code"),    str(gt["disease_code"])),
        "doctor_name":   (fields.get("doctor_name"),     str(gt["doctor_name"])),
        "med_name_1":    (fields["medications"][0] if len(fields["medications"]) > 0 else None, str(gt["med_name_1"])),
        "med_name_2":    (fields["medications"][1] if len(fields["medications"]) > 1 else None, str(gt["med_name_2"])),
        "med_name_3":    (fields["medications"][2] if len(fields["medications"]) > 2 else None, str(gt["med_name_3"])),
        "med_name_4":    (fields["medications"][3] if len(fields["medications"]) > 3 else None, str(gt["med_name_4"])),
        "med_name_5":    (fields["medications"][4] if len(fields["medications"]) > 4 else None, str(gt["med_name_5"])),
        "med_name_6":    (fields["medications"][5] if len(fields["medications"]) > 5 else None, str(gt["med_name_6"])),
        "med_name_7":    (fields["medications"][6] if len(fields["medications"]) > 6 else None, str(gt["med_name_7"])),
        "inject_name_1": (fields["inject_info"][0] if len(fields["inject_info"]) > 0 else None, str(gt["inject_name_1"])),
        "inject_name_2": (fields["inject_info"][1] if len(fields["inject_info"]) > 1 else None, str(gt["inject_name_2"])),
        "inject_name_3": (fields["inject_info"][2] if len(fields["inject_info"]) > 2 else None, str(gt["inject_name_3"])),
    }

    results = []
    for field, (pred, gt_val) in comparisons.items():
        match = normalize(pred) == normalize(gt_val)
        results.append({"field": field, "pred": pred, "gt": gt_val, "match": match})

    df_eval = pd.DataFrame(results)
    correct = df_eval["match"].sum()
    total = len(df_eval)
    print(f"정확도: {correct}/{total} ({correct/total*100:.1f}%)\n")
    print(df_eval.to_string(index=False))
    return df_eval

print("평가 함수 정의 완료")

## 단건 테스트

> 주석 해제 후 실행하세요.

In [ ]:
# test_fname = "처방전_0062.png"
# test_path = os.path.join(INPUT_DIR, test_fname)
# img = cv2.imdecode(np.fromfile(test_path, dtype=np.uint8), cv2.IMREAD_COLOR)
# img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#
# result = call_clova_ocr(test_path)
# texts = parse_clova_result(result)
# fields = extract_fields(texts)
#
# # 필드 출력
# print("=== 추출 결과 ===")
# for k, v in fields.items():
#     if v:
#         print(f"[{k}]")
#         if isinstance(v, list):
#             for item in v: print(f"  {item}")
#         else:
#             print(f"  {v}")
#
# # 정확도 평가
# evaluate(fields, fields.get("facility_code", ""))
#
# # 시각화
# fig, axes = plt.subplots(1, 2, figsize=(20, 14))
# axes[0].imshow(img_rgb); axes[0].set_title("원본"); axes[0].axis("off")
# axes[1].imshow(img_rgb); axes[1].set_title("OCR 결과"); axes[1].axis("off")
# for t in texts:
#     verts = t["bbox"]
#     xs = [v["x"] for v in verts]; ys = [v["y"] for v in verts]
#     x_min, y_min = min(xs), min(ys)
#     w, h = max(xs)-x_min, max(ys)-y_min
#     axes[1].add_patch(patches.Rectangle((x_min, y_min), w, h, linewidth=1, edgecolor="blue", facecolor="none"))
#     axes[1].text(x_min, y_min-2, t["text"], fontsize=5, color="blue")
# plt.tight_layout()
# plt.savefig(os.path.join(RESULT_DIR, f"{test_fname.replace('.png', '')}_clova_viz.png"), dpi=150, bbox_inches="tight")
# plt.show()

## 전체 배치 평가

> 주석 해제 후 실행하세요. Clova API 호출 비용 발생.

In [ ]:
# png_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".png")])
# print(f"총 {len(png_files)}개 파일")
#
# all_results = []
# for fname in png_files:
#     img_path = os.path.join(INPUT_DIR, fname)
#     result = call_clova_ocr(img_path)
#     texts = parse_clova_result(result)
#     fields = extract_fields(texts)
#
#     facility_code = fields.get("facility_code", "")
#     gt_row = df_gt[df_gt["facility_code"].astype(str) == str(facility_code)]
#     if len(gt_row) == 0:
#         print(f"정답 못 찾음: {fname}")
#         continue
#
#     gt = gt_row.iloc[0]
#     row = {"파일명": fname}
#     for field in ["facility_code", "facility_name", "patient_name", "patient_ssn", "disease_code", "doctor_name"]:
#         pred = fields.get(field)
#         row[field] = normalize(pred) == normalize(str(gt[field]))
#     for k in range(1, 8):
#         pred = fields["medications"][k-1] if len(fields["medications"]) >= k else None
#         row[f"med_name_{k}"] = normalize(pred) == normalize(str(gt[f"med_name_{k}"]))
#     for k in range(1, 4):
#         pred = fields["inject_info"][k-1] if len(fields["inject_info"]) >= k else None
#         row[f"inject_name_{k}"] = normalize(pred) == normalize(str(gt[f"inject_name_{k}"]))
#
#     all_results.append(row)
#     print(f"완료: {fname}")
#
# df_batch = pd.DataFrame(all_results)
# csv_out = os.path.join(RESULT_DIR, "batch_eval_results.csv")
# df_batch.to_csv(csv_out, index=False, encoding="utf-8-sig")
#
# bool_cols = [c for c in df_batch.columns if c != "파일명"]
# acc = df_batch[bool_cols].mean().mean() * 100
# print(f"\n전체 평균 정확도: {acc:.1f}%")
# print(f"결과 저장 → {csv_out}")